# Employee Attrition Prediction: Model Development.

### 4.1. Objective

The objective of this phase is to develop classification models capable of predicting whether an employee is likely to leave the organization.

Building on the findings from Exploratory Data Analysis and the machine-learning-ready datasets prepared during Phase 3, multiple classification algorithms will be trained and compared.

The modelling process will focus on developing a range of algorithms with different learning approaches, including:

- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- K-Nearest Neighbors
- Support Vector Machine

Because employee attrition is an imbalanced classification problem, class imbalance will be explicitly considered during model development.

Model evaluation and detailed interpretation will be conducted separately in Phase 5.

### 4.2. Import Libraries

In [10]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

### 4.3. Recreate Phase 3 Machine-Learning-Ready Dataset

Since each Jupyter notebook has its own execution environment, variables created in the Phase 3 notebook are not automatically available in the Phase 4 notebook.

To ensure reproducibility, the Phase 3 preprocessing workflow is recreated here using the same cleaned dataset, feature exclusions, train/test split, and preprocessing configuration.

The test set remains unseen during preprocessing. The preprocessing transformer is fitted only on the training data and then applied to the test data.

In [11]:
# Load cleaned Phase 1 dataset
df = pd.read_csv("clean_employee_attrition.csv")

# Create binary target
df['Attrition_Binary'] = df['Attrition'].map({
    'No': 0,
    'Yes': 1
})

# Separate features and target
X = df.drop(columns=['Attrition', 'Attrition_Binary'])
y = df['Attrition_Binary']

# Remove identifier and non-informative variables
irrelevant_cols = [
    'EmployeeNumber',
    'EmployeeCount',
    'StandardHours',
    'Over18'
]

X = X.drop(columns=irrelevant_cols)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Identify feature types
numerical_features = X_train.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=['object']
).columns.tolist()

# Build preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            numerical_features
        ),
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                drop='first'
            ),
            categorical_features
        )
    ]
)

# Fit ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using the fitted training transformer
X_test_processed = preprocessor.transform(X_test)

# Get feature names
feature_names = preprocessor.get_feature_names_out()

# Convert processed matrices to DataFrames
X_train_processed = pd.DataFrame(
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed.toarray()
    if hasattr(X_test_processed, "toarray")
    else X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Phase 3 preprocessing successfully recreated.")

Phase 3 preprocessing successfully recreated.


### 4.4. Verify Phase 3 Machine-Learning-Ready Data

The modelling stage uses the processed training and testing datasets produced in Phase 3.

No additional preprocessing or train/test splitting is performed in this phase. This ensures that the modelling workflow remains consistent with the leakage-aware preprocessing strategy established previously.

In [13]:
print("Training features:", X_train_processed.shape)
print("Testing features:", X_test_processed.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

print("\nTraining missing values:",
      X_train_processed.isnull().sum().sum())

print("Testing missing values:",
      X_test_processed.isnull().sum().sum())

Training features: (1176, 44)
Testing features: (294, 44)
Training target: (1176,)
Testing target: (294,)

Training missing values: 0
Testing missing values: 0


### 4.5. Target Class Distribution

Employee attrition is an imbalanced binary classification problem because substantially more employees remained with the organization than left.

This imbalance is important because a model that predominantly predicts the majority class could achieve high accuracy while performing poorly at identifying employees who are at risk of attrition.

Therefore, model development will consider class imbalance explicitly. Class weighting will be used for algorithms that support it, while balanced sample weights will be considered for Gradient Boosting.

Model performance will ultimately be assessed using multiple metrics, with particular attention to Recall, F1-score and ROC-AUC rather than accuracy alone.

In [14]:
class_distribution = pd.DataFrame({'Count': y_train.value_counts().sort_index(),'Percentage': (y_train.value_counts(normalize=True).sort_index().mul(100).round(2))})

class_distribution.index = ['Stayed (0)', 'Left (1)']

class_distribution

,Count,Percentage
Stayed (0),986,83.84
Left (1),190,16.16


### 4.6. Candidate Classification Models

Six classification algorithms spanning different learning paradigms will be developed to provide a broad comparison of different modelling approaches:

- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- K-Nearest Neighbors
- Support Vector Machine

The models represent different learning approaches, including linear, tree-based, ensemble, distance-based and margin-based methods.

All models will use the same training dataset and the same processed features to ensure a fair comparison.

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000,class_weight='balanced',random_state=RANDOM_STATE),

    'Decision Tree': DecisionTreeClassifier(class_weight='balanced',max_depth=6,random_state=RANDOM_STATE),

    'Random Forest': RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=RANDOM_STATE),

    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),

    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=9),

    'Support Vector Machine': SVC(kernel='rbf',probability=True,class_weight='balanced',random_state=RANDOM_STATE)
}

print('Models defined successfully:')
for name in models:
    print(f'- {name}')

Models defined successfully:
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- K-Nearest Neighbors
- Support Vector Machine


#### Class Imbalance Handling

Because employees who leave represent the minority class, class imbalance was considered during model development.

Class weighting was applied to Logistic Regression, Decision Tree, Random Forest and Support Vector Machine. This increases the relative penalty associated with incorrectly classifying observations from the minority class.

Gradient Boosting does not directly support a `class_weight` parameter, while K-Nearest Neighbors does not use class weighting in the same way. These models are therefore included without forced class weighting.

Hence the algorithms are allowed to be compared while acknowledging the characteristics of each modelling method.

### 4.7. Model Training

The candidate models will be trained exclusively on the training dataset.

The held-out test dataset will not be used during model development, model selection or performance comparison. It will remain reserved for the independent evaluation conducted in Phase 5.This separation will help to prevent test-set leakage and provides a more reliable estimate of model performance on unseen data.

In [17]:
fitted_models = {}

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    fitted_models[name] = model

print('All the six models trained successfully.')

All the six models trained successfully.


### 4.8. Cross-Validation for Model Development

Five-fold stratified cross-validation will be used to compare the candidate models using the training dataset.

Stratification ensures that each fold maintains a similar proportion of employees who stayed and employees who left.

The following metrics will be examined:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

Because attrition is an imbalanced classification problem, Recall, F1-score and ROC-AUC will receive particular attention.

The held-out test set remains untouched and will be used only during Phase 5.

In [18]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)

scoring = {'accuracy': 'accuracy','precision': 'precision','recall': 'recall','f1': 'f1','roc_auc': 'roc_auc'}

cv_results = []

for name, model in models.items():

    scores = cross_validate(model,X_train_processed,y_train,cv=cv,scoring=scoring,n_jobs=-1,return_train_score=False    )

    cv_results.append({'Model': name,'Accuracy': scores['test_accuracy'].mean(),'Precision': scores['test_precision'].mean(),'Recall': scores['test_recall'].mean(),
        'F1-Score': scores['test_f1'].mean(),'ROC-AUC': scores['test_roc_auc'].mean()})

cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values('ROC-AUC',ascending=False).reset_index(drop=True)

cv_results_df.round(4)

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.7466,0.3614,0.7263,0.4812,0.8268
1,Support Vector Machine,0.8393,0.5040,0.6053,0.5484,0.8115
2,Gradient Boosting,0.8750,0.7497,0.3474,0.4692,0.8090
3,Random Forest,0.8605,0.8873,0.1632,0.2716,0.8025
4,K-Nearest Neighbors,0.8469,0.8333,0.0632,0.1143,0.7226
5,Decision Tree,0.7517,0.3132,0.4263,0.3570,0.5925


In [19]:
# To add the model ranking
cv_results_df['Rank'] = (cv_results_df['ROC-AUC'].rank(ascending=False, method='min').astype(int))

cv_results_df = cv_results_df.sort_values('Rank')

cv_results_df.round(4)

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Rank
0,Logistic Regression,0.7466,0.3614,0.7263,0.4812,0.8268,1
1,Support Vector Machine,0.8393,0.5040,0.6053,0.5484,0.8115,2
2,Gradient Boosting,0.8750,0.7497,0.3474,0.4692,0.8090,3
3,Random Forest,0.8605,0.8873,0.1632,0.2716,0.8025,4
4,K-Nearest Neighbors,0.8469,0.8333,0.0632,0.1143,0.7226,5
5,Decision Tree,0.7517,0.3132,0.4263,0.3570,0.5925,6


In [20]:
# To identify the strongest model for each metric 

best_by_metric = {'Accuracy': cv_results_df.loc[cv_results_df['Accuracy'].idxmax(), 'Model'],

    'Precision': cv_results_df.loc[cv_results_df['Precision'].idxmax(), 'Model'],

    'Recall': cv_results_df.loc[cv_results_df['Recall'].idxmax(), 'Model'],

    'F1-Score': cv_results_df.loc[cv_results_df['F1-Score'].idxmax(), 'Model'],

    'ROC-AUC': cv_results_df.loc[cv_results_df['ROC-AUC'].idxmax(), 'Model']}

pd.DataFrame(best_by_metric.items(),columns=['Metric', 'Best Model'])

,Metric,Best Model
0,Accuracy,Gradient Boosting
1,Precision,Random Forest
2,Recall,Logistic Regression
3,F1-Score,Support Vector Machine
4,ROC-AUC,Logistic Regression


### 4.9. Model Development Findings

Five-fold stratified cross-validation was used to compare six classification algorithms using only the training dataset.

The results demonstrate clear differences in model behaviour across the evaluation metrics.

- **Logistic Regression** achieved the highest ROC-AUC (0.8268), Recall (0.7263), and the highest F1-score among the six models (0.4812). Its relatively high Recall indicates that it identified a larger proportion of employees who actually belong to the attrition class. However, this came with lower Precision (0.3614) and Accuracy (0.7466), meaning that the model also produced a relatively high number of false-positive attrition predictions.

- **Support Vector Machine** achieved the second-highest ROC-AUC (0.8115) and a Recall of 0.6053. It also achieved the highest F1-score overall (0.5484). This indicates a relatively stronger balance between Precision and Recall compared with several of the other models.

- **Gradient Boosting** achieved the highest Accuracy (0.8750) and the second-highest Precision (0.7497). However, its Recall was considerably lower at 0.3474, indicating that a substantial proportion of employees who actually left would not be identified by the model.

- **Random Forest** achieved the highest Precision (0.8873), but its Recall was only 0.1632. This indicates that although its positive predictions were highly precise, the model identified relatively few employees who actually left.

- **K-Nearest Neighbors** produced high Accuracy (0.8469) and Precision (0.8333), but its Recall was only 0.0632 and its F1-score was 0.1143. This suggests that its strong accuracy was largely driven by its ability to classify the majority class rather than effectively identifying employees at risk of attrition.

- **Decision Tree** produced the lowest ROC-AUC (0.5925) among the six models, suggesting relatively weak discrimination between employees who stayed and those who left.

Overall, the results demonstrate why accuracy alone is insufficient for evaluating this employee attrition problem. Since identifying employees who may be at risk of leaving is a key objective, Recall, F1-score and ROC-AUC are particularly important considerations.

Based on cross-validation performance, Logistic Regression and Support Vector Machine emerge as strong candidates for further evaluation, while Gradient Boosting remains a useful candidate because of its high Accuracy and Precision.

The held-out test dataset has not been used during this model development comparison and remains reserved for independent evaluation in Phase 5.

### 4.10. Creating the top-candidate table

In [23]:
top_candidates = cv_results_df[cv_results_df['Model'].isin(['Logistic Regression','Support Vector Machine','Gradient Boosting'])][
    ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]

top_candidates.round(4)

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.7466,0.3614,0.7263,0.4812,0.8268
1,Support Vector Machine,0.8393,0.5040,0.6053,0.5484,0.8115
2,Gradient Boosting,0.8750,0.7497,0.3474,0.4692,0.8090


### 4.11. Models Selected for Phase 5 Evaluation

Based on the cross-validation results, three models were selected for further evaluation using the held-out test dataset:

1. Logistic Regression
2. Support Vector Machine
3. Gradient Boosting

The selection was based on complementary strengths rather than accuracy alone.

- Logistic Regression was selected because it achieved the strongest ROC-AUC and Recall, making it particularly relevant to the objective of identifying employees who may be at risk of attrition.

- Support Vector Machine was selected because it achieved the strongest F1-score and the second-highest ROC-AUC, indicating a comparatively balanced performance between Precision and Recall.

- Gradient Boosting was selected because it achieved the highest Accuracy and strong Precision, providing an alternative model that performs particularly well in correctly classifying positive predictions.

The remaining models will not be prioritised for final evaluation because their cross-validation results were less competitive across the metrics most relevant to this business problem.

In [24]:
selected_model_names = ['Logistic Regression','Support Vector Machine','Gradient Boosting']

selected_models = {name: fitted_models[name]
    for name in selected_model_names
}

print("Selected models for Phase 5:")
for name in selected_models:
    print("-", name)

Selected models for Phase 5:
- Logistic Regression
- Support Vector Machine
- Gradient Boosting


In [26]:
# To save the selected models

import joblib

for name, model in selected_models.items():
    filename = (name.lower().replace(" ", "_").replace("-", "_"))

    joblib.dump(model,f"{filename}_phase4_model.pkl")

print("Selected Phase 4 models saved successfully.")

Selected Phase 4 models saved successfully.


In [27]:
X_test.to_csv("X_test_phase3.csv", index=False)
y_test.to_csv("y_test_phase3.csv", index=False)

print("Test data saved successfully.")

Test data saved successfully.


In [28]:
feature_names = preprocessor.get_feature_names_out()

pd.Series(feature_names).to_csv("feature_names_phase3.csv",index=False,header=["Feature"])

print("Feature names saved successfully.")

Feature names saved successfully.


### 4.12. Conclusion

Six classification models were developed and compared using five-fold stratified cross-validation on the training dataset.

The models included Logistic Regression, Support Vector Machine, Gradient Boosting, Random Forest, K-Nearest Neighbors and Decision Tree.

The cross-validation results showed that no single model dominated every metric.

Logistic Regression achieved the highest ROC-AUC (0.8268) and Recall (0.7263), making it particularly effective at identifying employees belonging to the attrition class. Support Vector Machine achieved the highest F1-score (0.5484), while Gradient Boosting achieved the highest Accuracy (0.8750) and strong Precision (0.7497).

The results also demonstrated the limitations of relying on Accuracy alone. Random Forest and K-Nearest Neighbors achieved relatively high Accuracy and Precision but recorded substantially lower Recall, indicating weaker performance in identifying employees who actually leave.

Based on the development results, Logistic Regression, Support Vector Machine and Gradient Boosting were selected as candidate models for formal evaluation.

The held-out test dataset was not used during model comparison or selection. It remains completely unseen and will be used in Phase 5 for independent model evaluation.